# Tema 14 — Seguimiento de objetos (tracking) con YOLO y BoT-SORT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-07/Tema-14/Tema_14.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En este notebook usamos un detector **YOLO** junto con el algoritmo de seguimiento **BoT-SORT** para detectar personas en un video y **asignarles un identificador (ID) estable** que se mantiene cuadro a cuadro, incluso ante oclusiones.

> 💡 Si lo abres en Colab, ejecuta primero la celda **Setup para Google Colab** (instala `ultralytics`). En entornos locales esa celda no hace nada y puedes saltarla.

> ⚙️ El código **detecta el entorno automáticamente**: en **local** muestra el seguimiento en vivo con `cv2.imshow` (sal con la tecla `q`) y en **Colab** guarda el resultado en `salida_tracking.mp4` y lo reproduce en la última celda. En ambos casos necesitas el archivo de video de prueba (`MOT17 04 FRCNN raw.mp4`); en Colab súbelo desde el panel de archivos de la izquierda.

In [ ]:
# === Setup para Google Colab ===
# Esta celda instala las dependencias que NO vienen preinstaladas en Colab.
# En entornos locales (VS Code / Jupyter) se ignora; si ya tienes ultralytics
# instalado también puedes saltarla.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # ultralytics -> YOLO + trackers (BoT-SORT). torch y opencv ya vienen en Colab.
    !pip install -q ultralytics
    print("Setup de Colab completado.")
else:
    print("Entorno local detectado, no se requiere setup adicional.")

**Inicializar los parámetros y umbrales de seguimiento de objetos**

### ¿Qué hace este código?

Esto **no es código Python**, sino la **configuración del algoritmo de seguimiento BoT-SORT** en formato YAML. Por eso la celda empieza con la *magic* **`%%writefile custom_tracker.yaml`**, que **guarda todo el contenido en el archivo `custom_tracker.yaml`** (el mismo que carga la celda de tracking con `tracker="custom_tracker.yaml"`).

> ⚠️ Sin `%%writefile`, Jupyter intentaría ejecutar las líneas como Python y daría un error (`NameError: name 'botsort' is not defined`).

Cada parámetro controla cómo se asignan y conservan los IDs:

- **`track_high_thresh` / `track_low_thresh`**: umbrales de confianza para aceptar o recuperar detecciones.
- **`new_track_thresh`**: qué tan seguro debe estar el modelo para crear un **ID nuevo** (evita falsos positivos).
- **`track_buffer`**: cuántos cuadros "recuerda" a un objeto **oculto** antes de descartarlo (manejo de oclusiones).
- **`match_thresh`**: similitud necesaria para asociar una detección al mismo ID.
- **`with_reid` + `proximity_thresh` / `appearance_thresh`**: activan la **reidentificación visual**, comparando la apariencia para no cambiar de ID cuando un objeto reaparece.

In [ ]:
%%writefile custom_tracker.yaml
# Ultralytics AGPL-3.0 License
# BoT-SORT tracker defaults for mode="track"

tracker_type: botsort # Define el uso de características avanzadas de BoT-SORT
track_high_thresh: 0.5 # Umbral inicial; valores altos limpian los rastros falsos
track_low_thresh: 0.1 # Umbral secundario para recuperar detecciones débiles
new_track_thresh: 0.7 # Exigencia para iniciar un ID nuevo (evita falsos positivos)
track_buffer: 250 # Cantidad de frames que "recuerda" un objeto oculto (Oclusión)
match_thresh: 0.8 # Similitud requerida para asociar el mismo ID
fuse_score: True # Fusiona la confianza de detección con el movimiento

# BoT-SORT specifics
gmc_method: sift # Compensación de movimiento global (útil si la cámara se mueve)

# ReID model related thresh (Reidentificación visual)
proximity_thresh: 0.5 # Distancia máxima para considerar el ReID
appearance_thresh: 0.5 # Similitud visual requerida para no cambiar de ID
with_reid: True # Activa el uso de características visuales profundas
model: auto # Modelo automático para extraer características

**Seguimiento de objetos con asignación de cuadros delimitadores**

### ¿Qué hace este código?

Ejecuta el **bucle de seguimiento** sobre el video y funciona **tanto en local como en Colab**:

1. **Detecta el entorno** (`IN_COLAB`), porque `cv2.imshow` **no funciona en Colab**.
2. **Configura el dispositivo** (GPU si hay `cuda`, si no CPU) y carga el modelo **YOLO**.
3. Verifica que el video exista (si no, muestra un mensaje claro) y prepara un **`cv2.VideoWriter`** para guardar el resultado en `salida_tracking.mp4`.
4. Por cada cuadro (*frame*) hace un **recorte central** de 300×300 px y llama a `model.track(...)` con `persist=True` para que el tracker (**BoT-SORT**, definido en `custom_tracker.yaml`) **mantenga los IDs**; se filtran solo personas (`classes=[0]`).
5. Dibuja el **bounding box**, un **color único por ID** y la etiqueta con ID + confianza, y **escribe el cuadro en el video de salida**.
6. **En local** además muestra la ventana en vivo (`cv2.imshow`, salir con **`q`**). **En Colab** solo guarda el archivo, que se visualiza en la siguiente celda.

In [ ]:
import os
import sys
import cv2
import torch
from ultralytics import YOLO

# Detecta el entorno: en Colab no existe ventana gráfica (cv2.imshow no funciona).
IN_COLAB = "google.colab" in sys.modules

# 1. Configuración de Dispositivo y Modelo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolo26x.pt').to(device)

# 2. Configuración de Video (entrada y salida)
video_path = 'MOT17 04 FRCNN raw.mp4'
output_path = 'salida_tracking.mp4'   # video anotado que se genera (sirve en local y Colab)

if not os.path.exists(video_path):
    raise FileNotFoundError(
        f"No se encontró '{video_path}'. Súbelo al entorno "
        "(en Colab usa el panel de archivos de la izquierda) y vuelve a ejecutar esta celda."
    )

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video '{video_path}'.")

# Configuraciones de detección y tracking
SCORE_THRESH = 0.7
IOU_THRESH = 0.6
TARGET_CLASSES = [0]  # Solo personas

# 3. Parámetros del recorte central y del escritor de video de salida
crop_w, crop_h = 300, 300
fps = cap.get(cv2.CAP_PROP_FPS) or 25
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (crop_w, crop_h))

print(f"Iniciando Tracking en {device} ({'Colab' if IN_COLAB else 'local'})...")
if not IN_COLAB:
    print("Presiona 'q' en la ventana para salir.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 4. Recorte (Crop) Central para optimizar el procesamiento
    h_orig, w_orig = frame.shape[:2]
    start_x = max(0, w_orig // 2 - (crop_w // 2))
    start_y = max(0, h_orig // 2 - (crop_h // 2))
    frame_crop = frame[start_y:start_y+crop_h, start_x:start_x+crop_w]
    # Aseguramos el tamaño exacto del recorte (por si el frame es más pequeño)
    frame_crop = cv2.resize(frame_crop, (crop_w, crop_h))

    # 5. Tracking
    results = model.track(
        source=frame_crop,
        persist=True,
        iou=IOU_THRESH,      # Umbral de Intersection over Union para asociación
        imgsz=400,          # Resolución de inferencia
        classes=TARGET_CLASSES, # Filtrar solo personas
        tracker="custom_tracker.yaml", # Archivo de configuración del algoritmo de tracking
        device=device,
        verbose=False,
    )[0]

    # 6. Proyección de resultados si hay identificadores confirmados
    if results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy().astype(int)
        ids = results.boxes.id.cpu().numpy().astype(int)
        confs = results.boxes.conf.cpu().numpy()

        for box, obj_id, conf in zip(boxes, ids, confs):
            if conf > SCORE_THRESH:
                xmin, ymin, xmax, ymax = box
                # --- Generación de color dinámico único basado en el ID ---
                color = (int((obj_id * 50) % 255), 255, int((obj_id * 30) % 255))
                # Dibujar Rectángulo y metadatos (ID y Confianza)
                cv2.rectangle(frame_crop, (xmin, ymin), (xmax, ymax), color, 2)
                display_text = f"ID:{obj_id} Pers: {conf:.2f}"
                cv2.putText(frame_crop, display_text, (xmin, ymin - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # 7. Guardar el cuadro anotado en el video de salida (funciona en local y Colab)
    writer.write(frame_crop)

    # 8. Despliegue en pantalla SOLO en local (en Colab cv2.imshow no funciona)
    if not IN_COLAB:
        cv2.imshow('Real-Time Tracking', frame_crop)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# 9. Liberar recursos
cap.release()
writer.release()
if not IN_COLAB:
    cv2.destroyAllWindows()

print(f"Listo. Video anotado guardado en '{output_path}'.")

**Reproducir el video con el seguimiento**

### ¿Qué hace este código?

Muestra el video anotado (`salida_tracking.mp4`) **dentro del notebook**. Es especialmente útil en **Google Colab**, donde no hay ventana de `cv2.imshow`:

1. Lee el archivo de salida y lo codifica en **base64**.
2. Lo incrusta en un reproductor `<video>` con controles de reproducción.

> 💡 En local también funciona; si prefieres, puedes abrir `salida_tracking.mp4` directamente con tu reproductor de video.

In [ ]:
# Reproduce el video anotado dentro del notebook (ideal para Colab).
import os
from base64 import b64encode
from IPython.display import HTML, display

output_path = 'salida_tracking.mp4'

if not os.path.exists(output_path):
    print(f"Aún no existe '{output_path}'. Ejecuta primero la celda de tracking.")
else:
    mp4 = open(output_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
        <video width=400 controls>
            <source src="{data_url}" type="video/mp4">
        </video>
    '''))